# Drone Audio Best Models - Standalone Inference Walkthrough

This self-contained notebook shows the full inference path for the two current best models:

- `tiny_mlp_es.joblib`: best lightweight candidate
- `random_forest.joblib`: best professor-recall candidate

Important: the `.joblib` files do not accept raw audio directly. The required path is:

```text
WAV audio -> 1s windows -> spectral_mfcc_v1 raw features -> scaler -> model -> threshold
```


## 1. Install Runtime Dependencies

Run this once in the Python environment used by the notebook:

```bash
python -m pip install numpy scipy joblib scikit-learn
```

Then restart the notebook kernel if needed.


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import csv
import json
import math
import struct
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import joblib
import numpy as np
from scipy import fft as scipy_fft


SCRIPT_DIR = Path.cwd()
POSITIVE_LABEL = "drone"
FEATURE_SET = "spectral_mfcc_v1"
WINDOW_SECONDS = 1.0
STRIDE_SECONDS = 0.5
EPSILON = 1e-12
MFCC_COEFFICIENT_COUNT = 20
MEL_FILTER_COUNT = 40
LOW_BAND_HZ = (20.0, 500.0)
MID_BAND_HZ = (500.0, 2_000.0)
WAVE_FORMAT_PCM = 1
WAVE_FORMAT_IEEE_FLOAT = 3
WAVE_FORMAT_EXTENSIBLE = 65534

V1_BASIC_FEATURE_NAMES = (
    "mean",
    "mean_abs",
    "rms",
    "peak_abs",
    "zero_crossing_rate",
)

SPECTRAL_FEATURE_NAMES = (
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_rolloff_85_hz",
    "spectral_flatness",
    "spectral_contrast_db",
    "dominant_frequency_hz",
    "low_band_energy",
    "mid_band_energy",
    "high_band_energy",
    "low_total_ratio",
    "mid_total_ratio",
    "high_total_ratio",
    "low_mid_ratio",
    "low_high_ratio",
    "mid_high_ratio",
)

MFCC_FEATURE_NAMES = (
    tuple(f"mfcc_{index:02d}_mean" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"mfcc_{index:02d}_std" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"delta_mfcc_{index:02d}_mean" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
    + tuple(f"delta_mfcc_{index:02d}_std" for index in range(1, MFCC_COEFFICIENT_COUNT + 1))
)

FEATURE_NAMES = V1_BASIC_FEATURE_NAMES + SPECTRAL_FEATURE_NAMES + MFCC_FEATURE_NAMES


@dataclass(frozen=True)
class WavMetadata:
    path: Path
    format_tag: int
    decoded_format_tag: int
    channels: int
    sample_rate_hz: int
    sample_width_bytes: int
    bits_per_sample: int
    block_align: int
    frames: int
    duration_sec: float
    data_offset: int
    data_size_bytes: int



def _discover_wavs(dataset_root: Path, include_calibration: bool) -> list[Path]:
    files = sorted(path for path in dataset_root.rglob("*.wav") if path.is_file())
    if include_calibration:
        return files
    return [path for path in files if not path.name.lower().startswith("calib_")]


def _default_artifact_path(repo_relative: str, local_name: str) -> Path:
    repo_path = Path(repo_relative)
    if repo_path.exists():
        return repo_path
    return SCRIPT_DIR / local_name


def _extract_dataset_features(
    wav_files: Sequence[Path],
    *,
    progress_every: int,
) -> tuple[list[dict[str, Any]], np.ndarray, dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    features: list[tuple[float, ...]] = []
    started_at = time.monotonic()
    non_finite_replacements = 0
    skipped_files: list[dict[str, str]] = []
    total_files = len(wav_files)

    for file_index, path in enumerate(wav_files, start=1):
        try:
            samples, metadata = read_wav_mono_array(path)
        except Exception as error:
            skipped_files.append({"path": str(path), "error": str(error)})
            print(f"feature progress: skipped file={path} error={error}", file=sys.stderr, flush=True)
            continue
        window_size = int(round(WINDOW_SECONDS * metadata.sample_rate_hz))
        hop_size = int(round(STRIDE_SECONDS * metadata.sample_rate_hz))
        if window_size <= 0 or hop_size <= 0 or len(samples) < window_size:
            skipped_files.append({"path": str(path), "error": "shorter than one complete window"})
            continue
        starts = range(0, len(samples) - window_size + 1, hop_size)
        file_window_count = 0
        for start_index in starts:
            end_index = start_index + window_size
            vector, replacements = extract_feature_vector(
                samples[start_index:end_index],
                metadata.sample_rate_hz,
            )
            non_finite_replacements += replacements
            start_sec = start_index / metadata.sample_rate_hz
            end_sec = end_index / metadata.sample_rate_hz
            rows.append(
                {
                    "file": str(path),
                    "file_name": path.name,
                    "sample_rate_hz": metadata.sample_rate_hz,
                    "window_start_sec": start_sec,
                    "window_end_sec": end_sec,
                }
            )
            features.append(vector)
            file_window_count += 1
            if progress_every > 0 and len(rows) % progress_every == 0:
                _print_feature_progress(
                    len(rows),
                    file_index,
                    total_files,
                    started_at,
                    current_file=path,
                )
        print(
            "feature progress: "
            f"file={file_index}/{total_files} windows={file_window_count} "
            f"sample_rate={metadata.sample_rate_hz} path={path.name}",
            flush=True,
        )

    if not features:
        raise RuntimeError("no feature rows extracted")
    _print_feature_progress(len(rows), total_files, total_files, started_at, current_file=wav_files[-1])
    return (
        rows,
        np.asarray(features, dtype=np.float64),
        {
            "non_finite_replacements": non_finite_replacements,
            "skipped_files": skipped_files,
            "elapsed_seconds": time.monotonic() - started_at,
        },
    )


def _print_feature_progress(
    processed_windows: int,
    file_index: int,
    total_files: int,
    started_at: float,
    *,
    current_file: Path,
) -> None:
    elapsed = max(time.monotonic() - started_at, 0.0)
    rate = processed_windows / elapsed if elapsed > 0 else 0.0
    print(
        "feature progress: "
        f"windows={processed_windows} files={file_index}/{total_files} "
        f"rate={rate:.1f} windows/s elapsed={_format_duration(elapsed)} "
        f"current_file={current_file.name}",
        flush=True,
    )


def extract_feature_vector(samples: Any, sample_rate_hz: int) -> tuple[tuple[float, ...], int]:
    array = np.asarray(samples, dtype=np.float64)
    if array.ndim != 1:
        array = array.reshape(-1)
    if array.size == 0:
        raise ValueError("cannot extract features from an empty audio window")
    values = _v1_basic_features(array) + _spectral_features(array, sample_rate_hz) + _mfcc_features(array, sample_rate_hz)
    return _finite_tuple(values)


def _v1_basic_features(samples: np.ndarray) -> tuple[float, ...]:
    abs_samples = np.abs(samples)
    signs = samples >= 0
    crossings = np.count_nonzero(signs[1:] != signs[:-1]) if len(samples) > 1 else 0
    zero_crossing_rate = crossings / (len(samples) - 1) if len(samples) > 1 else 0.0
    return (
        float(np.mean(samples)),
        float(np.mean(abs_samples)),
        float(np.sqrt(np.mean(samples * samples))),
        float(np.max(abs_samples)),
        float(zero_crossing_rate),
    )


def _spectral_features(samples: np.ndarray, sample_rate_hz: int) -> tuple[float, ...]:
    windowed = samples * np.hanning(len(samples))
    magnitudes = np.abs(np.fft.rfft(windowed))
    power = magnitudes * magnitudes
    freqs = np.fft.rfftfreq(len(samples), d=1.0 / sample_rate_hz)
    total_power = float(np.sum(power))
    if total_power <= EPSILON:
        return (0.0,) * len(SPECTRAL_FEATURE_NAMES)

    centroid = float(np.sum(freqs * power) / total_power)
    bandwidth = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * power) / total_power))
    cumulative = np.cumsum(power)
    rolloff_index = int(np.searchsorted(cumulative, 0.85 * total_power, side="left"))
    rolloff_index = min(rolloff_index, len(freqs) - 1)
    rolloff = float(freqs[rolloff_index])
    flatness = float(np.exp(np.mean(np.log(power + EPSILON))) / (np.mean(power) + EPSILON))
    positive = magnitudes[magnitudes > EPSILON]
    if positive.size:
        low_percentile = float(np.percentile(positive, 5))
        high_percentile = float(np.percentile(positive, 95))
        spectral_contrast = 20.0 * math.log10((high_percentile + EPSILON) / (low_percentile + EPSILON))
    else:
        spectral_contrast = 0.0
    dominant_frequency = float(freqs[int(np.argmax(power))])

    low_energy = _band_energy(freqs, power, LOW_BAND_HZ[0], LOW_BAND_HZ[1])
    mid_energy = _band_energy(freqs, power, MID_BAND_HZ[0], MID_BAND_HZ[1])
    high_energy = _band_energy(freqs, power, MID_BAND_HZ[1], sample_rate_hz / 2.0)
    return (
        centroid,
        bandwidth,
        rolloff,
        flatness,
        spectral_contrast,
        dominant_frequency,
        low_energy,
        mid_energy,
        high_energy,
        low_energy / (total_power + EPSILON),
        mid_energy / (total_power + EPSILON),
        high_energy / (total_power + EPSILON),
        low_energy / (mid_energy + EPSILON),
        low_energy / (high_energy + EPSILON),
        mid_energy / (high_energy + EPSILON),
    )


def _mfcc_features(samples: np.ndarray, sample_rate_hz: int) -> tuple[float, ...]:
    frames = _framed_audio(samples, sample_rate_hz)
    frame_length = frames.shape[1]
    n_fft = _next_power_of_two(frame_length)
    window = np.hanning(frame_length)
    spectrum = np.fft.rfft(frames * window, n=n_fft, axis=1)
    power = (np.abs(spectrum) ** 2) / n_fft
    filters = _mel_filterbank(sample_rate_hz, n_fft, MEL_FILTER_COUNT)
    mel_energy = np.maximum(power @ filters.T, EPSILON)
    log_mel = np.log(mel_energy)
    coefficients = scipy_fft.dct(log_mel, type=2, norm="ortho", axis=1)[:, :MFCC_COEFFICIENT_COUNT]
    if coefficients.shape[0] > 1:
        deltas = np.diff(coefficients, axis=0)
    else:
        deltas = np.zeros_like(coefficients)
    return tuple(
        [float(value) for value in np.mean(coefficients, axis=0)]
        + [float(value) for value in np.std(coefficients, axis=0)]
        + [float(value) for value in np.mean(deltas, axis=0)]
        + [float(value) for value in np.std(deltas, axis=0)]
    )


def _framed_audio(samples: np.ndarray, sample_rate_hz: int) -> np.ndarray:
    frame_length = max(1, int(round(0.025 * sample_rate_hz)))
    hop_length = max(1, int(round(0.010 * sample_rate_hz)))
    if len(samples) < frame_length:
        samples = np.pad(samples, (0, frame_length - len(samples)))
    frame_starts = range(0, len(samples) - frame_length + 1, hop_length)
    frames = [samples[start : start + frame_length] for start in frame_starts]
    if not frames:
        frames = [np.pad(samples, (0, frame_length - len(samples)))]
    return np.vstack(frames)


def _mel_filterbank(sample_rate_hz: int, n_fft: int, mel_count: int) -> np.ndarray:
    min_mel = _hz_to_mel(0.0)
    max_mel = _hz_to_mel(sample_rate_hz / 2.0)
    mel_points = np.linspace(min_mel, max_mel, mel_count + 2)
    hz_points = _mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sample_rate_hz).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)
    filters = np.zeros((mel_count, n_fft // 2 + 1), dtype=np.float64)
    for index in range(1, mel_count + 1):
        left, center, right = bins[index - 1], bins[index], bins[index + 1]
        if center == left:
            center = min(center + 1, n_fft // 2)
        if right == center:
            right = min(right + 1, n_fft // 2)
        for bin_index in range(left, center):
            filters[index - 1, bin_index] = (bin_index - left) / max(center - left, 1)
        for bin_index in range(center, right):
            filters[index - 1, bin_index] = (right - bin_index) / max(right - center, 1)
    return filters


def _hz_to_mel(value: Any) -> Any:
    return 2595.0 * np.log10(1.0 + np.asarray(value) / 700.0)


def _mel_to_hz(value: Any) -> Any:
    return 700.0 * (10.0 ** (np.asarray(value) / 2595.0) - 1.0)


def _band_energy(freqs: np.ndarray, power: np.ndarray, low_hz: float, high_hz: float) -> float:
    mask = (freqs >= low_hz) & (freqs < high_hz)
    return float(np.sum(power[mask]))


def _finite_tuple(values: tuple[float, ...]) -> tuple[tuple[float, ...], int]:
    sanitized = []
    replacements = 0
    for value in values:
        if math.isfinite(value):
            sanitized.append(float(value))
        else:
            sanitized.append(0.0)
            replacements += 1
    return tuple(sanitized), replacements


def _next_power_of_two(value: int) -> int:
    return 1 << (value - 1).bit_length()


def _load_scaler(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("feature_set") != FEATURE_SET:
        raise ValueError(f"scaler feature_set must be {FEATURE_SET}")
    if tuple(data.get("feature_names", [])) != FEATURE_NAMES:
        raise ValueError("scaler feature_names do not match standalone extractor")
    if len(data.get("mean", [])) != len(FEATURE_NAMES) or len(data.get("std", [])) != len(FEATURE_NAMES):
        raise ValueError("scaler mean/std dimensions do not match feature count")
    return data


def _apply_scaler(features: np.ndarray, scaler: Mapping[str, Any]) -> np.ndarray:
    mean = np.asarray(scaler["mean"], dtype=np.float64)
    std = np.asarray(scaler["std"], dtype=np.float64)
    if features.ndim != 2:
        raise ValueError("features must be a 2D matrix")
    if features.shape[1] != len(mean) or len(mean) != len(std):
        raise ValueError("scaler dimensions do not match features")
    return (features - mean) / std


def _load_model(path: Path) -> dict[str, Any]:
    model = joblib.load(path)
    if not isinstance(model, dict):
        raise ValueError(f"{path}: model artifact must be a dict package")
    return model


def _validate_model_package(name: str, model: Mapping[str, Any]) -> None:
    required = {
        "feature_set",
        "feature_names",
        "labels",
        "threshold",
        "threshold_name",
        "estimator",
    }
    missing = sorted(required - set(model))
    if missing:
        raise ValueError(f"{name}: model package missing keys: {', '.join(missing)}")
    if model["feature_set"] != FEATURE_SET:
        raise ValueError(f"{name}: feature_set must be {FEATURE_SET}")
    if tuple(model["feature_names"]) != FEATURE_NAMES:
        raise ValueError(f"{name}: feature_names do not match standalone extractor")
    labels = model["labels"]
    if labels.get("positive") != POSITIVE_LABEL:
        raise ValueError(f"{name}: positive label must be {POSITIVE_LABEL}")


def _score_model(
    model_name: str,
    estimator: Any,
    features: np.ndarray,
    *,
    batch_size: int,
) -> np.ndarray:
    scores: list[float] = []
    started_at = time.monotonic()
    total = len(features)
    for start in range(0, total, batch_size):
        batch = features[start : start + batch_size]
        scores.extend(_score_estimator(estimator, batch))
        processed = min(start + batch_size, total)
        elapsed = max(time.monotonic() - started_at, 0.0)
        rate = processed / elapsed if elapsed > 0 else 0.0
        eta = (total - processed) / rate if rate > 0 else 0.0
        print(
            "score progress: "
            f"model={model_name} windows={processed}/{total} "
            f"rate={rate:.1f} windows/s elapsed={_format_duration(elapsed)} "
            f"eta={_format_duration(eta)}",
            flush=True,
        )
    return np.asarray(scores, dtype=np.float64)


def _score_estimator(estimator: Any, features: np.ndarray) -> list[float]:
    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(features)
        classes = list(getattr(estimator, "classes_", []))
        if POSITIVE_LABEL not in classes:
            raise ValueError("estimator classes do not include the positive label")
        positive_index = classes.index(POSITIVE_LABEL)
        return [float(value) for value in probabilities[:, positive_index]]
    if hasattr(estimator, "decision_function"):
        decision = np.asarray(estimator.decision_function(features), dtype=np.float64)
        if decision.ndim == 2:
            classes = list(getattr(estimator, "classes_", []))
            positive_index = classes.index(POSITIVE_LABEL) if POSITIVE_LABEL in classes else 1
            decision = decision[:, positive_index]
        return [float(1.0 / (1.0 + math.exp(-value))) for value in decision]
    raise ValueError("estimator does not support predict_proba or decision_function")


def _summarize_scores(
    rows: Sequence[Mapping[str, Any]],
    scores: np.ndarray,
    detections: np.ndarray,
    threshold: float,
) -> dict[str, Any]:
    file_count = len({str(row["file"]) for row in rows})
    detected_files = {
        str(row["file"])
        for row, detected in zip(rows, detections, strict=True)
        if bool(detected)
    }
    return {
        "threshold": threshold,
        "windows": int(len(rows)),
        "detected_windows": int(np.count_nonzero(detections)),
        "window_detection_rate": float(np.mean(detections)) if len(detections) else 0.0,
        "files": file_count,
        "files_with_detection": len(detected_files),
        "file_detection_rate": len(detected_files) / file_count if file_count else 0.0,
        "score_mean": float(np.mean(scores)) if len(scores) else None,
        "score_median": float(np.median(scores)) if len(scores) else None,
        "score_p05": float(np.quantile(scores, 0.05)) if len(scores) else None,
        "score_p95": float(np.quantile(scores, 0.95)) if len(scores) else None,
        "score_max": float(np.max(scores)) if len(scores) else None,
    }


def _write_window_csv(
    path: Path,
    rows: Sequence[Mapping[str, Any]],
    scores: np.ndarray,
    detections: np.ndarray,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "file",
                "file_name",
                "sample_rate_hz",
                "window_start_sec",
                "window_end_sec",
                "score",
                "detected",
            ],
        )
        writer.writeheader()
        for row, score, detected in zip(rows, scores, detections, strict=True):
            writer.writerow(
                {
                    "file": row["file"],
                    "file_name": row["file_name"],
                    "sample_rate_hz": row["sample_rate_hz"],
                    "window_start_sec": f"{float(row['window_start_sec']):.3f}",
                    "window_end_sec": f"{float(row['window_end_sec']):.3f}",
                    "score": f"{float(score):.9f}",
                    "detected": int(bool(detected)),
                }
            )


def read_wav_metadata(path: Path) -> WavMetadata:
    with path.open("rb") as file:
        if file.read(4) != b"RIFF":
            raise ValueError(f"{path}: not a RIFF file")
        file.seek(8)
        if file.read(4) != b"WAVE":
            raise ValueError(f"{path}: not a WAVE file")

        fmt_payload: bytes | None = None
        data_offset: int | None = None
        data_size: int | None = None

        while True:
            header = file.read(8)
            if not header:
                break
            if len(header) != 8:
                raise ValueError(f"{path}: truncated chunk header")
            chunk_id, chunk_size = struct.unpack("<4sI", header)
            chunk_data_offset = file.tell()
            if chunk_id == b"fmt ":
                fmt_payload = file.read(chunk_size)
            elif chunk_id == b"data":
                data_offset = chunk_data_offset
                data_size = chunk_size
                file.seek(chunk_size, 1)
            else:
                file.seek(chunk_size, 1)
            if chunk_size % 2:
                file.seek(1, 1)

    if fmt_payload is None:
        raise ValueError(f"{path}: missing fmt chunk")
    if data_offset is None or data_size is None:
        raise ValueError(f"{path}: missing data chunk")
    if len(fmt_payload) < 16:
        raise ValueError(f"{path}: invalid fmt chunk")

    format_tag, channels, sample_rate_hz, _, block_align, bits_per_sample = struct.unpack(
        "<HHIIHH",
        fmt_payload[:16],
    )
    decoded_format_tag = _decoded_format_tag(format_tag, fmt_payload, path)
    if decoded_format_tag not in {WAVE_FORMAT_PCM, WAVE_FORMAT_IEEE_FLOAT}:
        raise ValueError(f"{path}: unsupported WAV format tag {format_tag}")
    if channels < 1:
        raise ValueError(f"{path}: WAV must have at least one channel")
    if sample_rate_hz <= 0:
        raise ValueError(f"{path}: invalid sample rate")
    if bits_per_sample % 8 != 0:
        raise ValueError(f"{path}: unsupported bit depth {bits_per_sample}")
    sample_width_bytes = bits_per_sample // 8
    frames = data_size // block_align
    return WavMetadata(
        path=path,
        format_tag=format_tag,
        decoded_format_tag=decoded_format_tag,
        channels=channels,
        sample_rate_hz=sample_rate_hz,
        sample_width_bytes=sample_width_bytes,
        bits_per_sample=bits_per_sample,
        block_align=block_align,
        frames=frames,
        duration_sec=frames / sample_rate_hz,
        data_offset=data_offset,
        data_size_bytes=data_size,
    )


def read_wav_mono_array(path: Path) -> tuple[np.ndarray, WavMetadata]:
    metadata = read_wav_metadata(path)
    with metadata.path.open("rb") as file:
        file.seek(metadata.data_offset)
        data = file.read(metadata.data_size_bytes)
    if len(data) != metadata.data_size_bytes:
        raise ValueError(f"{path}: truncated audio data")

    if metadata.decoded_format_tag == WAVE_FORMAT_IEEE_FLOAT:
        if metadata.sample_width_bytes == 4:
            array = np.frombuffer(data, dtype="<f4")
        elif metadata.sample_width_bytes == 8:
            array = np.frombuffer(data, dtype="<f8")
        else:
            raise ValueError(f"{metadata.path}: unsupported IEEE float width {metadata.sample_width_bytes}")
        return _mono_array(array, metadata.channels), metadata

    if metadata.sample_width_bytes == 1:
        array = (np.frombuffer(data, dtype=np.uint8).astype(np.float64) - 128.0) / 128.0
    elif metadata.sample_width_bytes == 2:
        array = np.frombuffer(data, dtype="<i2").astype(np.float64) / float(2**15)
    elif metadata.sample_width_bytes == 3:
        return np.asarray(_decode_frames(data, metadata), dtype=np.float64), metadata
    elif metadata.sample_width_bytes == 4:
        array = np.frombuffer(data, dtype="<i4").astype(np.float64) / float(2**31)
    else:
        raise ValueError(f"{metadata.path}: unsupported PCM width {metadata.sample_width_bytes}")
    return _mono_array(array, metadata.channels), metadata


def _mono_array(array: np.ndarray, channels: int) -> np.ndarray:
    if channels == 1:
        return array.astype(np.float64, copy=False)
    return array.reshape(-1, channels).mean(axis=1)


def _decoded_format_tag(format_tag: int, fmt_payload: bytes, path: Path) -> int:
    if format_tag != WAVE_FORMAT_EXTENSIBLE:
        return format_tag
    if len(fmt_payload) < 40:
        raise ValueError(f"{path}: invalid extensible fmt chunk")
    subformat = fmt_payload[24:40]
    return int.from_bytes(subformat[:2], byteorder="little")


def _decode_frames(data: bytes, metadata: WavMetadata) -> list[float]:
    samples: list[float] = []
    for frame_offset in range(0, len(data), metadata.block_align):
        channel_values = []
        for channel_index in range(metadata.channels):
            sample_offset = frame_offset + (channel_index * metadata.sample_width_bytes)
            raw = data[sample_offset : sample_offset + metadata.sample_width_bytes]
            if len(raw) != metadata.sample_width_bytes:
                continue
            channel_values.append(_decode_sample(raw, metadata))
        if channel_values:
            samples.append(sum(channel_values) / len(channel_values))
    return samples


def _decode_sample(raw: bytes, metadata: WavMetadata) -> float:
    if metadata.decoded_format_tag == WAVE_FORMAT_IEEE_FLOAT:
        if metadata.sample_width_bytes == 4:
            return float(struct.unpack("<f", raw)[0])
        if metadata.sample_width_bytes == 8:
            return float(struct.unpack("<d", raw)[0])
        raise ValueError(f"{metadata.path}: unsupported IEEE float width {metadata.sample_width_bytes}")
    if metadata.sample_width_bytes == 1:
        return (raw[0] - 128) / 128.0
    if metadata.sample_width_bytes == 2:
        return int.from_bytes(raw, byteorder="little", signed=True) / float(2**15)
    if metadata.sample_width_bytes == 3:
        sign_byte = b"\xff" if raw[2] & 0x80 else b"\x00"
        value = int.from_bytes(raw + sign_byte, byteorder="little", signed=True)
        return value / float(2**23)
    if metadata.sample_width_bytes == 4:
        return int.from_bytes(raw, byteorder="little", signed=True) / float(2**31)
    raise ValueError(f"{metadata.path}: unsupported PCM width {metadata.sample_width_bytes}")


def _require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise SystemExit(f"{label} does not exist or is not a file: {path}")


def _require_dir(path: Path, label: str) -> None:
    if not path.is_dir():
        raise SystemExit(f"{label} does not exist or is not a directory: {path}")


def _format_duration(seconds: float | None) -> str:
    if seconds is None:
        return "n/a"
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, rem = divmod(int(round(seconds)), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{rem:02d}s"
    return f"{minutes}m{rem:02d}s"


def _jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


from types import SimpleNamespace

infer = SimpleNamespace(
    FEATURE_SET=FEATURE_SET,
    FEATURE_NAMES=FEATURE_NAMES,
    _discover_wavs=_discover_wavs,
    read_wav_metadata=read_wav_metadata,
    _load_scaler=_load_scaler,
    _load_model=_load_model,
    _validate_model_package=_validate_model_package,
    _extract_dataset_features=_extract_dataset_features,
    _apply_scaler=_apply_scaler,
    _score_model=_score_model,
    _summarize_scores=_summarize_scores,
    _write_window_csv=_write_window_csv,
    _jsonable=_jsonable,
)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

print("Loaded self-contained inference helpers from this notebook")
print(f"Feature set: {infer.FEATURE_SET}")
print(f"Feature count: {len(infer.FEATURE_NAMES)}")


## 2. Configure Dataset And Artifact Paths

Download and extract the DroneNoise Database WAV files from:

https://salford.figshare.com/articles/dataset/DroneNoise_Database/22133411

Edit `DATASET_ROOT` below so it points to the extracted folder that contains the `.wav` files.

The model files and scaler are expected to be in the same folder where you extracted this bundle:

- `tiny_mlp_es.joblib`
- `random_forest.joblib`
- `v2_spectral_mfcc_scaler.json`

If your notebook kernel starts in a different working directory, set `ARTIFACT_DIR` to the folder containing those three files.


In [ ]:
# EDIT THIS: path to the extracted DroneNoise/Salford WAV folder.
# Example values:
#   DATASET_ROOT = Path("/Users/your_name/Downloads/dronenoise_database")
#   DATASET_ROOT = Path(r"C:/Users/your_name/Downloads/dronenoise_database")
DATASET_ROOT = Path("CHANGE_ME/path/to/dronenoise_database")

# Folder containing this notebook's model artifacts after unzip.
# If the notebook is opened from the extracted bundle folder, keep this as Path(".").
ARTIFACT_DIR = Path(".")

TINY_MLP_MODEL = ARTIFACT_DIR / "tiny_mlp_es.joblib"
RANDOM_FOREST_MODEL = ARTIFACT_DIR / "random_forest.joblib"
SCALER_PATH = ARTIFACT_DIR / "v2_spectral_mfcc_scaler.json"

INCLUDE_CALIBRATION = False
MAX_FILES = 0  # 0 means all WAV files. Use 5 for a quick smoke test.
OUT_DIR = Path("standalone_notebook_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for label, path in {
    "dataset": DATASET_ROOT,
    "tiny_mlp_es": TINY_MLP_MODEL,
    "random_forest": RANDOM_FOREST_MODEL,
    "scaler": SCALER_PATH,
}.items():
    print(f"{label:14s}: {path}  exists={path.exists()}")

if "CHANGE_ME" in str(DATASET_ROOT):
    raise ValueError("Please edit DATASET_ROOT so it points to the extracted DroneNoise WAV folder.")


## 3. Load Dataset Files

The model was trained on 1-second windows with 0.5-second stride. The notebook reads each WAV, downmixes to mono, and keeps the original sample rate for feature extraction.


In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder does not exist: {DATASET_ROOT}")

wav_files = infer._discover_wavs(DATASET_ROOT, INCLUDE_CALIBRATION)
if MAX_FILES:
    wav_files = wav_files[:MAX_FILES]

print(f"WAV files selected: {len(wav_files)}")
print("First files:")
for path in wav_files[:10]:
    meta = infer.read_wav_metadata(path)
    print(f"- {path.name}: sr={meta.sample_rate_hz}, channels={meta.channels}, duration={meta.duration_sec:.2f}s")


## 4. Load Scaler And Models

The scaler contains the train-set mean/std for the exact 100 feature columns. Each joblib file stores a model package with feature names, threshold, labels, and estimator.


In [ ]:
scaler = infer._load_scaler(SCALER_PATH)
models = {
    "tiny_mlp_es": infer._load_model(TINY_MLP_MODEL),
    "random_forest": infer._load_model(RANDOM_FOREST_MODEL),
}
for name, model in models.items():
    infer._validate_model_package(name, model)
    print(
        f"{name}: feature_set={model['feature_set']}, "
        f"threshold_name={model['threshold_name']}, threshold={float(model['threshold']):.9f}"
    )

print(f"Scaler feature_set={scaler['feature_set']}, train_rows={scaler.get('train_rows')}")


## 5. Extract `spectral_mfcc_v1` Features

This is the most important step. If someone extracts a different feature set, uses a different column order, or forgets the scaler, detections can fail even though the model weights are valid.


In [ ]:
rows, raw_features, extraction_summary = infer._extract_dataset_features(
    wav_files,
    progress_every=1000,
)
scaled_features = infer._apply_scaler(raw_features, scaler)

print(f"Raw feature matrix:    {raw_features.shape}")
print(f"Scaled feature matrix: {scaled_features.shape}")
print(f"Non-finite replacements: {extraction_summary['non_finite_replacements']}")
print(f"Skipped files: {len(extraction_summary['skipped_files'])}")


## 6. Run Inference

Scores are converted to detections using each model's stored `threshold_f2`.


In [ ]:
model_outputs = {}
for name, model in models.items():
    scores = infer._score_model(
        name,
        model["estimator"],
        scaled_features,
        batch_size=8192,
    )
    threshold = float(model["threshold"])
    detections = scores >= threshold
    summary = infer._summarize_scores(rows, scores, detections, threshold)
    infer._write_window_csv(OUT_DIR / f"{name}_windows.csv", rows, scores, detections)
    model_outputs[name] = {
        "scores": scores,
        "detections": detections,
        "summary": summary,
    }
    print(
        f"{name}: detected_windows={summary['detected_windows']}/{summary['windows']} "
        f"({summary['window_detection_rate']:.4f}), "
        f"files_with_detection={summary['files_with_detection']}/{summary['files']} "
        f"({summary['file_detection_rate']:.4f})"
    )


## 7. Save Summary JSON

The summary is useful to send back with test results. Per-window CSV files are written next to it.


In [ ]:
report = {
    "dataset_root": str(DATASET_ROOT),
    "wav_files": len(wav_files),
    "windows": len(rows),
    "feature_set": infer.FEATURE_SET,
    "feature_count": len(infer.FEATURE_NAMES),
    "include_calibration": INCLUDE_CALIBRATION,
    "extraction": extraction_summary,
    "models": {name: output["summary"] for name, output in model_outputs.items()},
}

summary_path = OUT_DIR / "summary.json"
summary_path.write_text(json.dumps(infer._jsonable(report), indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"Wrote {summary_path}")
print(json.dumps(infer._jsonable(report["models"]), indent=2, sort_keys=True))


## 8. Inspect Missed Windows

For a positive-only drone dataset, missed windows are useful for debugging. A file is usually considered detected if it has at least one detected window.


In [ ]:
for name, output in model_outputs.items():
    missed_indices = np.flatnonzero(~output["detections"])
    print(f"\n{name}: missed windows shown={min(10, len(missed_indices))}/{len(missed_indices)}")
    for index in missed_indices[:10]:
        row = rows[int(index)]
        score = float(output["scores"][int(index)])
        print(
            f"  {row['file_name']} "
            f"{float(row['window_start_sec']):.1f}-{float(row['window_end_sec']):.1f}s "
            f"score={score:.6f}"
        )


## 9. Interpreting Playback Tests

The direct WAV-file test above checks the model and feature pipeline. If direct WAV files detect but a speaker/microphone playback test has no detections, the likely issue is the playback/recording path: gain, clipping, echo, background noise, microphone frequency response, resampling, or using a feature pipeline that differs from this notebook.
